# GPR-FDTD-FWI experiment update_codex

Plain-English research update after reviewing `docs/experiments`, `docs/papers`, and `outputs/experiments`.

Generated on June 1, 2026. Scope: completed numbered runs `001` through `275`, plus the active/incomplete run `276` that is running in another terminal.


## One-page answer

This project is a 2D ground-penetrating radar research pipeline. It simulates radar waves moving through concrete, compares simulated radar data against observed or synthetic data, and estimates where steel reinforcing bars are and how large they are.

The current scientific objective is no longer just "make the inversion run." The objective is:

1. Find rebar lateral position `x`, cover depth `z`, and radius.
2. Keep working under noise, source-wavelet mismatch, and multiple nearby rebars.
3. Report uncertainty honestly when two nearby answers fit almost equally well.

Where we started: a forward finite-difference time-domain simulator and an interview-style full-waveform inversion proof of concept. It could generate B-scans and run inversion logic, but radius recovery was fragile.

Main milestones so far: single-rebar radius recovery was stabilized with local grid polish and source profiling; weak objective branches were exposed with top-candidate margins; detector-seeded two-stage refinement was built; high-band final polish improved radius confidence; material/source uncertainty reporting was added; and the same reporting discipline was carried into multi-rebar coordinate optimization.

Where we are now: the active branch is a hard multi-rebar, variable-radius, close-spacing case. The latest completed evidence says a 40 mm transmitter/receiver offset with 5 scan positions resolves the rightmost close-spaced target reliably across seeds. A 3-source version fails. Run `276` is currently testing the in-between 4-source case.

Where we are going: let run `276` finish, aggregate the 3/4/5-source comparison, then decide the minimum acquisition density needed for the close-50 mm multi-rebar case before scaling the pipeline further.


## Plain-language glossary

| Term | Plain meaning in this repo |
| --- | --- |
| GPR | Ground-penetrating radar. A transmitter sends an electromagnetic pulse into concrete, and a receiver records echoes. |
| FDTD | Finite-difference time-domain. This is the wave simulator. It advances electromagnetic fields step by step on a grid. |
| CPML | Convolutional perfectly matched layer. This is an absorbing boundary so simulated waves do not reflect from the edge of the grid. |
| GPU-CPML | The production GPU simulator with CPML absorbing boundaries. This is the main backend for heavy experiments. |
| FWI | Full-waveform inversion. This means changing model parameters until simulated waveforms match observed waveforms. |
| B-scan | A radar image made from many traces while scanning across the surface. Rebars appear as curved hyperbola-like reflections. |
| Tx/Rx | Transmitter/receiver. A larger Tx/Rx offset means the transmitter and receiver are farther apart. |
| Objective or misfit | A single number measuring how different simulated data are from observed data. Lower is better. |
| Least squares | The main objective: square the difference between simulated and observed traces, then sum it. |
| Source profiling | Fitting nuisance source parameters such as amplitude, time shift, and center-frequency scale so radius is not blamed for source error. |
| Confidence margin | How much better the best radius is than the next different radius. Small margin means the answer is fragile. |
| Ambiguity interval | The range of x, z, or radius values that fit almost as well as the best answer. This is reported instead of hiding uncertainty. |
| High-band polish | A final radius check using higher-frequency synthetic data, often 2.5 GHz, because higher frequencies carry more small-radius detail. |
| PEBDD | Progressively expanded bandwidths of data. Start with lower-frequency content and gradually include higher-frequency content. |
| W2 or optimal transport | Alternative waveform-distance ideas tested as possible objectives. They were useful as diagnostics but were not promoted as the final radius objective. |


## Experiment archive map

There are 276 numbered experiment folders under `outputs/experiments`. Runs `001` through `275` are completed enough to have artifacts or summaries. Run `276` has been allocated and is actively running.

The category index currently classifies the completed archive through `275` as 122 single-rebar runs, 80 multi-rebar runs, and 73 infrastructure/reporting runs. Run `276` is another multi-rebar coordinate optimizer run.

| Runs | Count | Main question | Bottom line |
| --- | ---: | --- | --- |
| `001-006` | 6 | What do the x/z/r objective landscapes look like? | Radius and depth are coupled; local landscapes are essential. |
| `007-023` | 17 | Can the single-rebar pipeline recover x, z, radius under noise? | Location works; continuous radius optimization drifts high; local grid polish fixes radius when the window is right. |
| `024-041` | 18 | Are trace-shift, bandpass, and cumulative-frequency objectives useful? | Low bands help basin finding but can dilute radius evidence. High-frequency content is needed for final radius. |
| `042-056` | 15 | Repair plotting, design spectrum experiments, test W2/optimal transport, wavelet mismatch, and material tradeoff. | W2 was not promoted for radius selection; source mismatch became a major risk; source profiling fixed tested mismatch cases. |
| `057-062` | 6 | Turn source profiling into a production-style local radius polish. | Radius 6 mm recovered under source mismatch/noise; wavefield animations were added for interpretation. |
| `063-080` | 18 | Move from one rebar to three rebars with confidence reporting. | Fixed-position common/per-target radius works; 24/24 local x/z/r cases are correct, but most margins are weak, so ambiguity intervals are mandatory. |
| `081-106` | 26 | Build the reporting-first multi-rebar coordinate optimizer. | One-step noisy coordinate updates recover truth; wider seeds need guarded revisits; high-band helps after geometry is already corrected. |
| `107-115` | 9 | Add B-scan detection before inversion. | A time-offset-aware hyperbola detector gives reliable x/z seed windows in tested single and multi-rebar cases. |
| `116-128` | 13 | Package detector-to-FWI two-stage single-rebar refinement. | Coarse-to-fine refinement cuts runtime; shallow 4 mm bars are correct but weak-confidence, requiring interval reporting. |
| `129-147` | 19 | Diagnose shallow-radius ambiguity with subcell geometry, multifrequency, denser acquisition, and 0.5 mm grid checks. | Subcell and denser sources help, but uncertainty remains for shallow small bars. |
| `148-163` | 16 | Add guarded polish and r=8 objective diagnostics. | Guarded polish keeps point accuracy but can lower confidence; amplitude fitting must stay because removing it biases radius. |
| `164-188` | 25 | Test high-band acquisition and package high-band final polish. | 2.5 GHz high-band polish strongly improves radius margins and works in packaged r=4 and r=8 flows. |
| `189-201` | 13 | Add material/source uncertainty reporting. | Nominal point estimates stay exact, but material/source assumptions widen the honest radius interval. |
| `202-215` | 14 | Return to multi-rebar coordinate diagnostics and detector assignments. | Assignment/reporting infrastructure was prepared for harder variable-radius scenes. |
| `216-257` | 42 | Variable-radius close-spacing multi-rebar pipeline with seeds 13, 21, 34. | Staged pipeline recovers truth after focused refinement; 7-source focused runs remove a 1 mm x ambiguity seen in 5-source rows. |
| `258-275` | 18 | Hard close-50 mm target: acquisition density, Tx/Rx offset, and top-candidate objective gaps. | 5 sources with 40 mm Tx/Rx offset works across seeds; 3 sources fails; run `276` is testing 4 sources. |
| `276` | active | Does 4 sources with 40 mm Tx/Rx offset bridge the gap between failing 3 sources and successful 5 sources? | Running now. No summary JSON yet. |


## Where we started

The project began as a working 2D electromagnetic simulator and inversion proof of concept:

- Forward simulation produced B-scans with rebar reflections.
- The adjoint-state full-waveform inversion path existed.
- CPU tests and GPU benchmark code existed.
- The original demo was closer to a three-rebar interview deliverable than a systematic research pipeline.

The first major research turn was to simplify the inverse problem to one circular rebar in 2D concrete. The unknowns were only:

```text
x center, z depth, radius
```

That simplification was important because the early optimizer could move toward the correct location but tended to overestimate radius. The research therefore shifted from "run an optimizer" to "understand the objective landscape and report uncertainty."


## Milestone 1: single-rebar radius became reliable enough to build on

The first single-rebar branch found the key failure: a broad continuous optimizer could land in a high-radius branch around 6.8 to 7.0 mm even when the true radius was 6.0 mm.

The fix was not a bigger global optimizer. The useful recipe became:

1. Get x and z into the right local window.
2. Evaluate a small grid of plausible radius values.
3. Fit source nuisance parameters so source error is not mistaken for radius error.
4. Report the best radius, the next radius, and the margin.

The source profiling experiments were decisive. Raw fixed-source least-squares could pick the wrong radius under wavelet mismatch. Once amplitude, time shift, and center-frequency scale were profiled, the tested mismatch cases returned to the true radius.

<img src="../../outputs/experiments/055_wavelet_mismatch_radius_amp_time_freqfit/figures/wavelet_mismatch_radius_profiles.png" width="850">

Caption: Experiment `055` shows radius profiles after amplitude, time-shift, and frequency-scale profiling. The minimum returns to the true 6.0 mm radius across source mismatch cases.


## Milestone 2: several tempting objective ideas were tested and not forced

The paper-backed ideas were useful, but not all became production choices.

- Progressive bandwidth and frequency weighting helped diagnose where radius information lives. Low frequencies were useful for basin finding but too weak for final radius decisions.
- W2 or optimal-transport style distances behaved well on simple shifted traces but did not produce a useful rebar-radius landscape. They were not promoted as final objectives.
- Full wavefield-reconstruction inversion and neural implicit inversion were documented as larger future branches, not the immediate fix for this low-dimensional problem.

The practical result was conservative: keep source-profiled least-squares as the main final objective, use frequency/bandwidth tools as diagnostics or staged helpers, and require candidate margins before making claims.

<img src="../../outputs/presentation_figures_single_rebar_next/pebdd_stage_progression.png" width="850">

Caption: Spectrum and bandwidth work led to a staged view: lower-band information can guide the search, but final radius evidence needs higher-frequency content and explicit margin checks.


## Milestone 3: detector-to-FWI became a packaged single-rebar pipeline

The next step was to stop assuming a known local window. A B-scan detector was added to find likely rebar windows from radar data.

The detector does not estimate radius. It only proposes x/z seed windows. Radius is still estimated by source-profiled FWI refinement.

The packaged single-rebar pipeline became:

```text
B-scan detector -> cheap 2 mm coarse screen -> 1 mm fine polish -> optional high-band polish -> optional material/source uncertainty report
```

Main outcomes:

- Single-rebar detector benchmark: 48/48 hits under the tested depth/radius/noise matrix.
- Source-mismatch detector benchmark: 48/48 hits.
- Two-stage refinement recovered the tested r=6, r=8, and shallow r=4 cases.
- Shallow r=4 cases were point-correct but weak-confidence until high-band final polish improved margins.
- Material/source uncertainty reporting made it clear that point radius and honest interval are not the same thing.

<img src="../../outputs/experiments/201_packaged_highband_material_uncertainty_r4_r8_178_200/figures/two_stage_material_uncertainty_summary.png" width="850">

Caption: Experiment `201` compares nominal high-band radius intervals against material/source-aware intervals. The optimizer point can be exact while the honest interval widens when material or source assumptions are allowed to vary.

<img src="../../outputs/experiments/194_single_rebar_r4_material_source_tradeoff_highband_subcell13_seed13/figures/true_r4_sigma1e7_observed_source_wavefield.gif" width="520">

Caption: Experiment `194` wavefield animation for the shallow 4 mm material/source tradeoff branch. It is included to show that the reports are tied back to simulated wave propagation, not only summary tables.


## Milestone 4: multi-rebar work kept the same reporting discipline

The multi-rebar branch started in controlled steps:

1. Fixed x/z positions, one common radius for all three rebars.
2. Sweep one target radius while holding the other two fixed.
3. Allow local x/z/r coupling for one target at a time.
4. Aggregate confidence over targets and noise seeds.

The important result was subtle:

```text
24/24 local target/case rows recovered true x, z, and radius,
but 22/24 rows were weak-confidence.
```

That means the model often chooses the right point, but a nearby deeper/larger-radius candidate can fit almost as well. This is why ambiguity intervals became a required output.

<img src="../../outputs/experiments/080_multi_rebar_stage7_ambiguity_interval_report/figures/candidate_confidence_margins.png" width="950">

Caption: Experiment `080` shows multi-rebar radius margins. Most rows are correct but weak. The right response is not to hide the weakness, but to report the ambiguity interval.


## Milestone 5: the coordinate optimizer became reporting-first, not blind search

The multi-rebar coordinate optimizer updates one target at a time. It is not a blind 9-parameter global optimizer. Each update writes:

- top candidates,
- best and next-radius margin,
- confidence label,
- ambiguity interval,
- selected source profile.

The first noisy compact-window tests recovered all three same-radius rebars across seeds. Wider seed-offset tests exposed an edge-target failure mode: the selected point could move to a deeper/larger-radius branch even when the true branch was close behind.

The remedy was a guarded revisit: if an edge update is weak and the ambiguity interval spans the true-radius branch, revisit that target after the other targets are corrected.

High-band objective variants helped after the neighboring geometry was already corrected. They did not replace guarded revisit when neighboring target estimates were still wrong. This is an important negative result.


## Milestone 6: detection and assignment moved the pipeline toward end-to-end use

The detector learned to account for source-time offset. Without that, it found the right x but biased depth. With an offset grid, the same detector found useful x/z windows.

The detector then worked on multi-rebar cases, including close spacing and variable depth. Later assignment reports connected detections to coordinate optimizer commands, so the multi-rebar pipeline could start from detected candidates rather than hard-coded truth windows.

<img src="../../outputs/experiments/216_detection_multi_rebar_variable_radius_close_spacing_source_mismatch_noise10/figures/detection_overlay.png" width="850">

Caption: Experiment `216` detector overlay for the variable-radius close-spacing branch. Colored hyperbola curves mark candidate rebar detections on the B-scan. This is the seed layer for later FWI refinement.


## Current main branch: variable-radius close-spacing multi-rebar

The recent multi-rebar branch is harder than the earlier same-radius case.

The close-60 mm setup used:

```text
x = [190, 250, 310] mm
z = [90, 90, 90] mm
radius = [5, 6, 8] mm
noise = 10%
source mismatch = frequency scale 1.1, time shift -50 ps, amplitude scale 1.1
```

The staged policy was:

1. Detector and assignment.
2. Location-only coordinate optimization, with radii held fixed.
3. Focused target-2 refinement for the large right bar.
4. Joint-radius profile with x/z fixed.

Results:

- Seeds 13 and 21 recovered the true radius tuple `[5, 6, 8]` in the joint-radius stage.
- Seed 34 initially had a 1 mm lateral ambiguity for the right target and a weak competing radius tuple.
- A 7-source focused refinement removed that 1 mm x ambiguity and brought seed 34 back to the true final tuple.

<img src="../../outputs/experiments/248_variable_radius_staged_pipeline_seed13_21_34_summary/figures/staged_variable_radius_pipeline_errors.png" width="850">

Caption: Experiment `248` summarizes staged error reduction for seeds 13, 21, and 34. Location-only reduces x/z errors but radius remains wrong; focused and joint stages are needed for the variable-radius case.


## Latest completed branch: close-50 mm spacing and acquisition design

The newest completed branch makes spacing even tighter:

```text
x = [190, 250, 300] mm
z = [90, 90, 90] mm
radius = [5, 6, 8] mm
```

The hard target is the rightmost bar at `x=300 mm`, `radius=8 mm`. The main ambiguity is a nearby competing branch around `x=299 mm` with a smaller radius.

Runs `258-263` showed the problem: with normal acquisition settings, 5, 7, and 9 sources could give strong radius margins, but lateral x ambiguity could remain near the default 1.5% threshold.

<img src="../../outputs/experiments/263_lateral_gap_threshold_close50_vs_close60_summary/figures/lateral_neighbor_gap_thresholds.png" width="850">

Caption: Experiment `263` compares the true right-bar x position against the nearest-left competitor. Points above the dashed 1.5% line are separated enough to avoid being reported as ambiguous. Close-50 is harder than close-60.


## Latest completed finding: 40 mm Tx/Rx offset is a strong acquisition lever

Runs `267-273` tested a wider 40 mm transmitter/receiver offset for the close-50 case.

Completed evidence:

- With 5 sources and 40 mm Tx/Rx offset, seeds 13, 21, and 34 all prefer the true `x=300 mm` target over the `x=299 mm` competitor.
- The base objective clears the 1.5% ambiguity threshold in every tested row.
- The high-band diagnostic objective increases the gap even more.
- Aggregate `273` reports 8 strong confidence rows and 7/8 truth-geometry rows. The one non-truth row comes from an older no-offset/not-recorded comparison, not the new 40 mm offset seed replication.

<img src="../../outputs/experiments/271_close50_txrx40_seed_replication_summary/figures/txrx40_seed_gap_replication.png" width="900">

Caption: Experiment `271` is the cleanest current evidence. Positive bars mean the true `x=300 mm` candidate beats the `x=299 mm` competitor. All tested seeds and cases clear the ambiguity threshold with 5 sources and 40 mm Tx/Rx offset.


## Right now: 3 sources fails, 5 sources succeeds, 4 sources is running

Runs `274-275` tested whether the 40 mm Tx/Rx result can be made cheaper by reducing the number of scan positions.

Completed evidence:

- 3 sources with 40 mm Tx/Rx offset is not enough. Aggregate `275` has 6 rows: 4 weak rows and only 2 strong rows. The 3-source rows prefer `x=299 mm`, `r=7.5 mm`, not the true `x=300 mm`, `r=8.0 mm`.
- 5 sources with 40 mm Tx/Rx offset succeeds. The 5-source rows are strong, truth-geometry rows with no x ambiguity.
- Run `276` is currently testing 4 sources with the same 40 mm Tx/Rx offset to find the minimum reliable acquisition setting.

<img src="../../outputs/experiments/275_coordinate_confidence_aggregate_close50_txrx40_sources3_vs5_seed34/figures/coordinate_confidence_aggregate.png" width="850">

Caption: Experiment `275` directly compares 3-source and 5-source 40 mm-offset results. Red weak bars are the 3-source rows; green strong bars are the 5-source rows. This is why run `276` is testing 4 sources.


## Current objectives and scoring objects

If "current objects" means "current objective functions and report objects," this is the current stack.

| Object | Role now | Status |
| --- | --- | --- |
| Source-profiled least-squares | Main final misfit. It compares observed and simulated traces while fitting source amplitude/time/frequency nuisance parameters. | Workhorse objective. |
| Base coordinate objective | 1.5 GHz source-profiled least-squares over local x/z/r target grids. | Main coordinate optimizer score. |
| High-band diagnostic objective | A higher-frequency or high-band window used to test whether candidates separate better. | Useful diagnostic and final polish in single-rebar cases. Not a universal replacement. |
| Detector hyperbola score | Finds x/z seed windows from B-scans before FWI. | Good seed layer. Does not estimate radius. |
| Confidence report | Converts top candidates into strong/moderate/weak labels and ambiguity intervals. | Mandatory reporting layer. |
| Material/source uncertainty report | Compares nominal radius against radius intervals when material/source assumptions vary. | Reporting/calibration layer, not an optimizer replacement. |
| W2/optimal transport | Alternative waveform distance tested from papers. | Not promoted for final rebar radius selection. |
| PEBDD/frequency schedule | Lower-to-higher frequency strategy from papers. | Useful design principle; final radius still needs high-frequency evidence and margin checks. |


## Where this is going next

The immediate next decision depends on run `276`.

If 4 sources with 40 mm Tx/Rx offset succeeds:

- Aggregate sources 3, 4, and 5 for the close-50 target-2 case.
- Treat 4 sources as a candidate minimum acquisition setting.
- Replicate 4 sources across seeds before promoting it.

If 4 sources fails:

- Treat 5 sources with 40 mm Tx/Rx offset as the current minimum reliable setting for this close-50 synthetic case.
- Do not spend more time trying to under-sample unless a new acquisition idea is introduced.

After that, the next research steps are:

1. Update prose trackers for runs `202-276`, because recent output folders are ahead of `docs/experiments/47_detection_to_fwi_pipeline.md`.
2. Extend close-50 acquisition conclusions beyond target 2 and beyond seed 34.
3. Fold the successful acquisition setting back into the staged variable-radius pipeline, not just the target-2 diagnostic.
4. Keep interval reporting and nuisance-aware reporting in every final result.


## Selected figure and animation index

| Artifact | Why it is included |
| --- | --- |
| `outputs/experiments/055_wavelet_mismatch_radius_amp_time_freqfit/figures/wavelet_mismatch_radius_profiles.png` | Shows source profiling fixing a major single-rebar radius failure mode. |
| `outputs/presentation_figures_single_rebar_next/pebdd_stage_progression.png` | Summarizes the bandwidth/frequency stage idea. |
| `outputs/experiments/201_packaged_highband_material_uncertainty_r4_r8_178_200/figures/two_stage_material_uncertainty_summary.png` | Shows the difference between exact point estimates and honest uncertainty intervals. |
| `outputs/experiments/194_single_rebar_r4_material_source_tradeoff_highband_subcell13_seed13/figures/true_r4_sigma1e7_observed_source_wavefield.gif` | Representative wavefield animation for the material/source tradeoff branch. |
| `outputs/experiments/080_multi_rebar_stage7_ambiguity_interval_report/figures/candidate_confidence_margins.png` | Shows why multi-rebar outputs must report weak confidence and ambiguity intervals. |
| `outputs/experiments/216_detection_multi_rebar_variable_radius_close_spacing_source_mismatch_noise10/figures/detection_overlay.png` | Shows the detector seed layer for the current variable-radius close-spacing branch. |
| `outputs/experiments/248_variable_radius_staged_pipeline_seed13_21_34_summary/figures/staged_variable_radius_pipeline_errors.png` | Shows how the variable-radius pipeline reduces errors by stage. |
| `outputs/experiments/263_lateral_gap_threshold_close50_vs_close60_summary/figures/lateral_neighbor_gap_thresholds.png` | Shows why close-50 spacing is harder than close-60. |
| `outputs/experiments/271_close50_txrx40_seed_replication_summary/figures/txrx40_seed_gap_replication.png` | Shows the strongest completed evidence for 40 mm Tx/Rx offset. |
| `outputs/experiments/275_coordinate_confidence_aggregate_close50_txrx40_sources3_vs5_seed34/figures/coordinate_confidence_aggregate.png` | Shows why 3 sources is insufficient and why 4 sources is currently being tested. |


## Final plain-English status

The repo is in a much stronger place than the starting proof of concept. It now has a repeatable pattern:

```text
detect likely rebar windows,
refine geometry with source-profiled FWI,
check high-band or diagnostic objectives when radius confidence is weak,
report top candidates and intervals instead of only a best point.
```

The biggest remaining problem is not that the code cannot find correct answers in tested synthetic cases. The bigger problem is deciding when an answer is confident enough under tight spacing, noise, source mismatch, and nuisance material/source assumptions.

The current active experiment is exactly about that: finding the minimum acquisition setting that still separates the true close-spaced right rebar from a near-by competing branch.
